# **Mapping Topik BERTopic -> Aspek FiQA**

Memetakan tiap topik hasil BERTopic ke **4 kategori FiQA**: `Corporate`, `Stock`, `Market`, `Economy` (Maia et al. 2018; Lengkeek et al. 2023), lalu memberi **nama aspek** yang lebih spesifik.

- Input: `data/processed/sentences_with_topic.csv` + `data/processed/topic_info.csv`.
- Output: `data/processed/mapping_aspek.csv` berisi usulan mapping tiap topik untuk **dicek/diedit manual**.

Mapping ini **usulan awal berbasis keyword** - keputusan final tetap manual (baca kalimat representatif, gabungkan topik semakna). Topik `-1` (outlier) dan topik tak jelas -> kategori **Lainnya**.

In [ ]:
from pathlib import Path
import pandas as pd

sent = pd.read_csv("data/processed/sentences_with_topic.csv", dtype={"id_artikel": str})
info = pd.read_csv("data/processed/topic_info.csv")
print(f"Kalimat: {len(sent):,} | Topik: {len(info)}")
info.head()

## Usulan mapping tiap topik -> kategori FiQA + nama aspek

Dict di bawah = usulan awal (berdasarkan keyword). **Edit bebas** sesuai penilaian manual. Format: `topic_id: ("Kategori FiQA", "Nama Aspek")`.

In [ ]:
# USULAN MAPPING - silakan edit nilainya setelah membaca kalimat representatif
USULAN = {
    -1: ("Lainnya", "Lainnya / Outlier"),
    # --- MARKET ---
    0:  ("Market",    "Pergerakan IHSG"),
    4:  ("Market",    "Pergerakan Sektoral"),
    6:  ("Market",    "Nilai Transaksi Pasar"),
    12: ("Market",    "Level / Penutupan Indeks"),
    15: ("Market",    "Breadth Pasar (naik/turun)"),
    25: ("Market",    "Bursa Asia"),
    37: ("Market",    "Segmen Pasar (reguler/negosiasi)"),
    38: ("Market",    "Bursa Global (Asia-Pasifik)"),
    40: ("Market",    "Kapitalisasi Pasar"),
    41: ("Market",    "Bursa AS (Wall Street)"),
    # --- STOCK ---
    1:  ("Stock",     "Kontribusi Saham ke Indeks"),
    5:  ("Stock",     "Aksi Asing (net buy/sell)"),
    8:  ("Stock",     "Aksi Asing per Emiten"),
    20: ("Stock",     "Aksi Asing (saham spesifik)"),
    31: ("Stock",     "Saham Spesifik (BUMI)"),
    32: ("Stock",     "Penggerak Indeks (perbankan)"),
    33: ("Stock",     "Saham Spesifik (Barito)"),
    35: ("Stock",     "Net Buy/Sell Emiten"),
    44: ("Stock",     "Saham Perbankan"),
    45: ("Stock",     "Saham Spesifik (komoditas)"),
    # --- CORPORATE ---
    2:  ("Corporate", "Strategi & Ekspansi Bisnis"),
    7:  ("Corporate", "Dividen & RUPS"),
    9:  ("Corporate", "Bank Syariah"),
    10: ("Corporate", "Layanan Digital & Produk"),
    13: ("Corporate", "Kredit & Pembiayaan"),
    14: ("Corporate", "Kinerja Keuangan / Laba"),
    16: ("Corporate", "Manajemen & Kepemimpinan"),
    21: ("Corporate", "Dividen (nominal/interim)"),
    22: ("Corporate", "Produk Tabungan / Rekening"),
    23: ("Corporate", "Pertumbuhan Kredit & DPK"),
    26: ("Corporate", "Buyback Saham"),
    34: ("Corporate", "Penyaluran Kredit"),
    36: ("Corporate", "Kualitas Kredit (NPL)"),
    # --- ECONOMY ---
    3:  ("Economy",   "Indeks Global (MSCI)"),
    18: ("Economy",   "Kebijakan The Fed"),
    19: ("Economy",   "Pertumbuhan Ekonomi"),
    24: ("Economy",   "Suku Bunga BI"),
    27: ("Economy",   "Nilai Tukar Rupiah"),
    28: ("Economy",   "Nilai Tukar (Dolar)"),
    29: ("Economy",   "Perang Dagang / Tarif"),
    30: ("Economy",   "Geopolitik"),
    42: ("Economy",   "Komoditas Emas"),
    43: ("Economy",   "Komoditas Minyak"),
    # --- LAINNYA (perlu review manual) ---
    11: ("Lainnya",   "Kutipan Percakapan"),
    17: ("Lainnya",   "Profil Tokoh / Kekayaan"),
    39: ("Lainnya",   "Pengantar Tabel"),
}
print(f"Topik dipetakan: {len(USULAN)}")

## Bangun tabel mapping untuk dicek (gabung dengan keyword + contoh kalimat)

In [ ]:
info = info.copy()
info["kategori_fiqa"] = info["Topic"].map(lambda t: USULAN.get(int(t), ("Lainnya", ""))[0])
info["nama_aspek"]   = info["Topic"].map(lambda t: USULAN.get(int(t), ("Lainnya", ""))[1])

# cek: adakah topik yang belum punya usulan?
belum = info.loc[info["nama_aspek"] == "", "Topic"].tolist()
print("Topik tanpa usulan mapping:", belum if belum else "tidak ada")

mapping = info[["Topic", "Count", "kategori_fiqa", "nama_aspek", "Keywords",
                "Representative_1", "Representative_2", "Representative_3"]].copy()
mapping = mapping.sort_values(["kategori_fiqa", "Topic"]).reset_index(drop=True)
mapping

## Ringkasan distribusi per kategori & per aspek

In [ ]:
tot = info["Count"].sum()
print("Distribusi kalimat per KATEGORI FiQA:")
rk = info.groupby("kategori_fiqa")["Count"].sum().sort_values(ascending=False)
for k, v in rk.items():
    print(f"  {k:10} {v:>6,} ({v/tot*100:4.1f}%)")
print(f"\nJumlah aspek (tanpa Lainnya): {mapping.loc[mapping.kategori_fiqa != 'Lainnya', 'nama_aspek'].nunique()}")

## Terapkan ke tiap kalimat & simpan output untuk dicek

In [ ]:
peta = {int(t): v for t, v in USULAN.items()}
sent = sent.copy()
sent["kategori_fiqa"] = sent["topic"].map(lambda t: peta.get(int(t), ("Lainnya", ""))[0])
sent["nama_aspek"]   = sent["topic"].map(lambda t: peta.get(int(t), ("Lainnya", ""))[1])

OUT = Path("data/processed")
# 1) tabel mapping per topik (ringkas, untuk dicek manual)
mapping.to_csv(OUT / "mapping_aspek.csv", index=False, encoding="utf-8-sig")
# 2) kalimat + aspek (hasil penerapan mapping)
sent.to_csv(OUT / "sentences_with_aspect.csv", index=False, encoding="utf-8-sig")
print(f"[1] mapping_aspek.csv -> {len(mapping)} topik")
print(f"[2] sentences_with_aspect.csv -> {len(sent):,} kalimat")

## Langkah manual berikutnya

1. Buka `mapping_aspek.csv`, baca `Keywords` + 3 kalimat representatif tiap topik, **koreksi** `kategori_fiqa` / `nama_aspek` bila perlu.
2. **Gabungkan** aspek yang maknanya sama (mis. beberapa 'Aksi Asing' jadi satu) agar menyusut ke **~10-15 aspek**.
3. Pindahkan topik tak jelas ke **Lainnya**.
4. Tulis **definisi 1 kalimat + contoh batas** tiap aspek di `daftar_aspek.md`.
5. **Validasi penetapan aspek pada sampel**, lalu **kunci daftar aspek** sebelum anotasi sentimen.